# 🤖 Grant Agent — Complete Colab Runner

**What this notebook does:** Runs the entire Grant Agent platform — backend API, AI agents, and Playwright browser automation — directly inside Google Colab. Your database, uploaded documents, and browser screenshots are stored on your Google Drive so they survive session resets.

---

## How to use this notebook

Run the cells **in order**, top to bottom. Read the comment block at the top of each cell — it tells you exactly what the cell does and what to expect.

| Cell | Purpose | Must run? |
|------|---------|----------|
| **Cell 1** | Mount Drive, clone repo, install everything | ✅ Always |
| **Cell 2** | Set your email, Gemini API key, GCP project | ✅ Always |
| **Cell 3** | Start the Grant Agent server + get your public URL | ✅ Always |
| **Cell 4** | Write your email into the org profile so the bot uses it | Optional (run once) |
| **Cell 5** | Test browser automation on a real grant website | Optional |
| **Cell 6** | Scrape and discover a new grant from any URL | Optional |
| **Cell 7** | Deploy to Google Cloud Run (production, always-on) | Optional |

---

## ⚠️ Session limits
| Colab plan | Max session | Note |
|------------|-------------|------|
| Free | ~12 hours | Disconnects if you close the browser tab |
| Pro | ~24 hours | Can keep running in background |

**Data is never lost** because everything is on Google Drive. When a session ends, just re-run Cells 1 → 2 → 3 and you are back up in under 2 minutes (deps are already installed on Drive).

In [ ]:
# ================================================================
# CELL 1 — Mount Google Drive & Install All Dependencies
#
# What it does:
#   1. Mounts your Google Drive at /content/drive
#   2. Creates four folders under MyDrive/GrantAgentData:
#        db/          — SQLite database file (persists across sessions)
#        uploads/     — documents you upload (PDFs, etc.)
#        screenshots/ — browser automation screenshots
#        logs/        — server log files
#   3. Clones the Grant Agent repo from GitHub (branch: feature/grant-agent)
#      OR does a git pull if it was cloned in a previous session
#   4. Installs Python packages from requirements.txt for both
#      the API and the browser worker
#   5. Installs Playwright + downloads Chromium browser (~150 MB)
#   6. Installs cloudflared (creates a public HTTPS URL for your server)
#
# First run: takes ~4-6 minutes
# Subsequent runs: ~30-60 seconds (packages already cached)
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

# ── Persistent folders on Google Drive ───────────────────────────
DATA_DIR = '/content/drive/MyDrive/GrantAgentData'
for sub in ['db', 'uploads', 'screenshots', 'logs']:
    os.makedirs(f'{DATA_DIR}/{sub}', exist_ok=True)
print(f'✅ Google Drive folders ready at: {DATA_DIR}')
print(f'   db/          → where your grant database lives')
print(f'   uploads/     → your uploaded documents')
print(f'   screenshots/ → browser bot screenshots')
print(f'   logs/        → server output logs')

# ── Clone or update the repository ───────────────────────────────
REPO_DIR = '/content/grant-agent'
REPO_URL = 'https://github.com/ademikelz19-hub/cookie-project.git'
BRANCH   = 'feature/grant-agent'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('\n🔄 Repository already cloned — pulling latest changes...')
    result = subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=REPO_DIR,
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
else:
    print(f'\n📥 Cloning repository (branch: {BRANCH})...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
    print('✅ Repository cloned')

# ── Install Python packages ───────────────────────────────────────
print('\n📦 Installing API dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', f'{REPO_DIR}/grant-agent-api/requirements.txt'], check=True)

print('📦 Installing browser worker dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', f'{REPO_DIR}/grant-agent-worker/requirements.txt'], check=True)

# ── Install Playwright Chromium browser ──────────────────────────
print('\n🎭 Installing Playwright Chromium (needed for browser automation)...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'],
               check=True)
print('✅ Chromium installed — the browser bot can now open real websites')

# ── Install cloudflared (creates your public URL) ─────────────────
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('\n🌐 Installing cloudflared (public URL tunnel)...')
    subprocess.run([
        'bash', '-c',
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
        '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared'
    ], check=True)
    print('✅ cloudflared installed')
else:
    print('\n✅ cloudflared already installed')

print('\n' + '='*60)
print('✅ CELL 1 COMPLETE — Ready to configure (run Cell 2 next)')
print('='*60)

In [ ]:
# ================================================================
# CELL 2 — Your Personal Configuration
#
# What to fill in:
#   GEMINI_API_KEY  — your free Gemini key from Google AI Studio
#                     Get one at: https://aistudio.google.com/app/apikey
#                     Without this, the AI agents run in offline/demo mode
#                     (they still work, just no real AI responses)
#
#   YOUR_EMAIL      — your real email address
#                     The Playwright browser bot reads this when it detects
#                     an email field on a real grant application form and
#                     fills it in automatically. It also gets stored in your
#                     organisation profile in the database.
#
#   GCP_PROJECT_ID  — only needed if you plan to deploy to Google Cloud
#                     in Cell 7. Leave as-is if you only want Colab.
#
# Nothing is sent anywhere when you run this cell. These variables
# just stay inside the Colab notebook session.
# ================================================================

# ── EDIT THESE THREE VALUES ──────────────────────────────────────

GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'
# ^ paste your key between the quotes above

YOUR_EMAIL = 'your.real.email@example.com'
# ^ the browser bot fills this on real grant forms

GCP_PROJECT_ID = 'grant-agent-gcp'
# ^ your Google Cloud project ID (only for Cell 7 / Cloud Run deploy)

# ── These are auto-set, no need to change ────────────────────────
import secrets
API_PORT   = 8000
GCP_REGION = 'us-central1'
JWT_SECRET = secrets.token_hex(32)   # random secure key, regenerated each session

# ── Confirm your settings ────────────────────────────────────────
print('⚙️  Configuration saved for this session:')
print(f'   Your email      : {YOUR_EMAIL}')
print(f'   Gemini API key  : {GEMINI_API_KEY[:10]}...{GEMINI_API_KEY[-4:]}' if len(GEMINI_API_KEY) > 20
      else f'   Gemini API key  : {GEMINI_API_KEY}  ← looks like a placeholder, AI will run in demo mode')
print(f'   GCP project     : {GCP_PROJECT_ID}  (only used in Cell 7)')
print(f'   API port        : {API_PORT}')
print(f'   JWT secret      : [auto-generated, {len(JWT_SECRET)} chars]')
print()
if YOUR_EMAIL == 'your.real.email@example.com':
    print('⚠️  WARNING: You have not changed YOUR_EMAIL yet.')
    print('   The browser bot will fill "your.real.email@example.com" on grant forms.')
    print('   Edit line 40 above and re-run this cell.')
else:
    print('✅ Ready — run Cell 3 to start the server')

In [ ]:
# ================================================================
# CELL 3 — Start the Grant Agent Server + Open a Public URL
#
# What it does:
#   1. Writes a .env file with your settings from Cell 2.
#      The database URL points to Google Drive so data persists:
#        sqlite:////content/drive/MyDrive/GrantAgentData/db/grant_agent.db
#
#   2. Starts the FastAPI backend server on port 8000 using uvicorn.
#      On first ever boot, the server automatically:
#        - creates all database tables
#        - seeds two example organisations:
#            Lioris (Web3 startup, Nigeria)
#            Princess Sara Foundation (nonprofit, education, Nigeria)
#        - seeds five example grants (GREEN, GREEN, YELLOW, ORANGE, RED stage)
#        - creates a demo admin account:
#            email:    demo@grantagent.ai
#            password: password123
#
#   3. Starts a cloudflared tunnel that gives you a public HTTPS URL
#      like https://abc123xyz.trycloudflare.com
#      You can open this URL from your phone, laptop, anywhere.
#      The URL changes every time you restart Cell 3.
#
#   4. Starts a keep-alive thread that pings the server every 30 seconds
#      so Colab does not disconnect you due to inactivity.
#
# KEEP THIS CELL RUNNING the entire time you use Grant Agent.
# If you interrupt it, the server stops and you need to re-run it.
# Server logs are written to: MyDrive/GrantAgentData/logs/api_server.log
# ================================================================

import os, subprocess, threading, time, re, sys
import urllib.request

REPO_DIR = '/content/grant-agent'
DATA_DIR = '/content/drive/MyDrive/GrantAgentData'
api_path    = f'{REPO_DIR}/grant-agent-api'
worker_path = f'{REPO_DIR}/grant-agent-worker'

# ── Write the .env file ───────────────────────────────────────────
env_text = f"""# Grant Agent — auto-generated by Colab notebook
# Database on Google Drive (persists across sessions)
DATABASE_URL=sqlite:///{DATA_DIR}/db/grant_agent.db

# Auth
SECRET_KEY={JWT_SECRET}
ALGORITHM=HS256
ACCESS_TOKEN_EXPIRE_MINUTES=10080

# Gemini AI — 3 models tried in order if one fails
GEMINI_API_KEY={GEMINI_API_KEY}
PRIMARY_GEMINI_MODEL=gemini-2.0-flash
FALLBACK_GEMINI_MODEL=gemini-1.5-flash
SECONDARY_FALLBACK_GEMINI_MODEL=gemini-1.5-flash-8b
GEMINI_MAX_RETRIES=4
GEMINI_RETRY_BASE_SECONDS=2
GEMINI_REQUEST_TIMEOUT_SECONDS=45

# Environment
ENVIRONMENT=colab
PROJECT_NAME=Grant Agent
LOG_LEVEL=INFO

# File storage on Drive
UPLOAD_DIR={DATA_DIR}/uploads
SCREENSHOTS_DIR={DATA_DIR}/screenshots

# CORS — allow all origins so frontend and API docs work
CORS_ORIGINS=["*"]
"""

env_path = f'{REPO_DIR}/.env'
with open(env_path, 'w') as f:
    f.write(env_text)
print('✅ .env file written to:', env_path)

# ── Set PYTHONPATH so Python finds both the API and worker packages ─
env = os.environ.copy()
env['PYTHONPATH'] = f'{api_path}:{worker_path}'
for p in [api_path, worker_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Start the FastAPI server ──────────────────────────────────────
log_file = open(f'{DATA_DIR}/logs/api_server.log', 'a')
print(f'\n🚀 Starting Grant Agent server on port {API_PORT}...')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(API_PORT), '--log-level', 'info'],
    cwd=api_path, env=env,
    stdout=log_file, stderr=subprocess.STDOUT
)

# Wait up to 30 seconds for the server to become healthy
print('⏳ Waiting for server to start (seeding database on first run)...')
time.sleep(5)
server_up = False
for attempt in range(15):
    try:
        urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=3)
        server_up = True
        print(f'✅ Server is up! (attempt {attempt+1})')
        break
    except Exception:
        time.sleep(2)

if not server_up:
    print('❌ Server did not start within 30 seconds.')
    print(f'   Check the log file: {DATA_DIR}/logs/api_server.log')
    print('   Or run: !tail -50 "/content/drive/MyDrive/GrantAgentData/logs/api_server.log"')
    raise RuntimeError('Grant Agent server failed to start')

# ── Start cloudflared tunnel ──────────────────────────────────────
print('\n🌐 Starting cloudflared tunnel (getting your public URL)...')
tunnel_url = None
tunnel_lines = []

def _read_tunnel(proc):
    global tunnel_url
    for line in iter(proc.stdout.readline, b''):
        s = line.decode('utf-8', errors='ignore').strip()
        tunnel_lines.append(s)
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', s)
        if m and not tunnel_url:
            tunnel_url = m.group(0)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=_read_tunnel, args=(tunnel_proc,), daemon=True).start()

for _ in range(30):       # wait up to 60 seconds for the URL
    if tunnel_url:
        break
    time.sleep(2)

# ── Print your access details ─────────────────────────────────────
print()
print('='*65)
if tunnel_url:
    print('🌐  Your Grant Agent is LIVE at:')
    print(f'\n      {tunnel_url}')
    print(f'\n  Open this in any browser — phone, laptop, anywhere.')
    print(f'\n  📖  API documentation : {tunnel_url}/docs')
    print(f'  ❤️   Health check      : {tunnel_url}/health')
else:
    print('⚠️  Could not detect tunnel URL. Last tunnel output:')
    for ln in tunnel_lines[-5:]:
        print(f'     {ln}')

print()
print('  🔑  Default login credentials:')
print('      Email    : demo@grantagent.ai')
print('      Password : password123')
print()
print('  📁  Your data is stored at:')
print(f'      {DATA_DIR}/')
print()
print('  ⚠️  This URL changes on every restart.')
print('      To get a new one: Runtime → Interrupt → re-run Cell 3.')
print('='*65)

# ── Keep-alive: ping every 30s to stop Colab idle disconnect ──────
def _keep_alive():
    while True:
        try:
            urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=5)
        except Exception:
            pass
        time.sleep(30)

threading.Thread(target=_keep_alive, daemon=True).start()
print('\n💓 Keep-alive active — pinging server every 30 seconds')

In [ ]:
# ================================================================
# CELL 4 — Write YOUR Email into the Organisation Profiles
#
# What it does:
#   The Playwright browser bot (Cell 5) fills grant application forms
#   using data from your organisation profile in the database.
#   When it finds an email field on a real grant site, it reads
#   org.email from the database and fills it in.
#
#   This cell:
#     1. Logs into the Grant Agent API using the demo account
#     2. Gets your list of organisations (Lioris + Princess Sara Foundation)
#     3. Updates the email field on each one to YOUR_EMAIL (set in Cell 2)
#
#   Run this once after Cell 3. After that the email is saved in the
#   database on Drive — it persists across sessions automatically.
#
# Expected output:
#   ✅ Logged in
#   📋 Found 2 organisation(s):
#      [0] Lioris — current email: contact@lioris.network
#      [1] Princess Sara Foundation — current email: grants@princesssarafoundation.org
#   ✅ Lioris → email updated to: your.real.email@example.com
#   ✅ Princess Sara Foundation → email updated to: your.real.email@example.com
# ================================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

# Login to get a JWT token
login_data = urllib.parse.urlencode({
    'username': 'demo@grantagent.ai',
    'password': 'password123'
}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
print('✅ Logged in to Grant Agent API')

headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# Get all organisations
req = urllib.request.Request(f'{BASE}/organisations', headers=headers)
with urllib.request.urlopen(req) as r:
    orgs = json.loads(r.read())

print(f'\n📋 Found {len(orgs)} organisation(s):')
for i, o in enumerate(orgs):
    print(f'   [{i}] {o["organisation_name"]} — current email: {o.get("email", "(none)")}')

# Update each org's email
print(f'\n🔄 Updating email to: {YOUR_EMAIL}')
for org in orgs:
    # Copy all org fields (excluding relational sub-objects) and patch email
    skip_keys = {'id', 'owner_id', 'created_at', 'updated_at',
                 'founders', 'team_members', 'metrics', 'documents'}
    payload = {k: v for k, v in org.items() if k not in skip_keys}
    payload['email'] = YOUR_EMAIL

    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        f'{BASE}/organisations/{org["id"]}',
        data=data, headers=headers, method='PUT'
    )
    with urllib.request.urlopen(req) as r:
        updated = json.loads(r.read())
    print(f'   ✅ {updated["organisation_name"]} → email: {updated["email"]}')

print(f'\n🎯 Done. The browser bot will now fill "{YOUR_EMAIL}" on all grant forms.')
print('   This is saved in the database on Google Drive — no need to run again.')

In [ ]:
# ================================================================
# CELL 5 — Test Browser Automation on a Real Grant Website
#
# What it does:
#   Sends the Playwright browser bot to the grant URL you specify.
#   The bot will:
#     1. Open the URL in a headless Chromium browser
#     2. Scan the page for form fields
#     3. Fill in every field it can match to your org profile:
#        - email field        → your email (YOUR_EMAIL from Cell 2)
#        - org name field     → "Lioris" or "Princess Sara Foundation"
#        - phone field        → org phone number
#        - website field      → org website URL
#        - country field      → Nigeria
#        - problem statement  → org's problems_addressed text
#        - solution field     → org's products/technology_solution text
#        - budget field       → 50,000 (default)
#     4. STOP immediately if it detects:
#        - CAPTCHA  → takes a screenshot and pauses
#        - OTP      → takes a screenshot and pauses
#        - 2FA      → takes a screenshot and pauses
#        - Login gate → stops and reports
#
# The bot NEVER submits the form without your review.
# Screenshots are saved to: MyDrive/GrantAgentData/screenshots/
#
# HOW TO USE:
#   1. Change TEST_GRANT_URL to a real grant application page
#   2. Change ORG_INDEX to 0 (Lioris) or 1 (Princess Sara Foundation)
#   3. Run the cell and read the output at the bottom
#
# Note: This creates a BrowserSession record in the database.
#   The session log shows every event (field filled, gate detected, etc.)
# ================================================================

import urllib.request, urllib.parse, json, time

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── CONFIGURE THIS ────────────────────────────────────────────────
TEST_GRANT_URL = 'https://www.tonyelumelufoundation.org/apply'
# ^ Replace with any real grant application URL
# Examples:
#   'https://apply.tonyelumelufoundation.org/'
#   'https://grants.example.com/apply'

ORG_INDEX = 0
# ^ 0 = Lioris (Web3 startup), 1 = Princess Sara Foundation (nonprofit)
# ─────────────────────────────────────────────────────────────────

# Login
login_data = urllib.parse.urlencode({'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# Get org
req = urllib.request.Request(f'{BASE}/organisations', headers=headers)
with urllib.request.urlopen(req) as r:
    orgs = json.loads(r.read())
org = orgs[ORG_INDEX]
ORG_ID = org['id']
print(f'🏢 Organisation : {org["organisation_name"]}')
print(f'   Email         : {org.get("email", "(none — run Cell 4 first)")}')
print(f'   Type          : {org.get("organisation_type")}')
print(f'   Stage         : {org.get("stage")}')

# Get first available grant from the database
req = urllib.request.Request(f'{BASE}/grants', headers=headers)
with urllib.request.urlopen(req) as r:
    grants = json.loads(r.read())
if not grants:
    print('\n❌ No grants in database. Run Cell 6 to scrape a grant URL first.')
    raise SystemExit(0)
GRANT_ID = grants[0]['id']
print(f'\n📄 Grant        : {grants[0]["grant_name"]}')
print(f'   Funder        : {grants[0].get("funder")}')

# Create an application record linking the org and the grant
app_payload = json.dumps({'org_id': ORG_ID, 'grant_id': GRANT_ID}).encode()
req = urllib.request.Request(f'{BASE}/applications', data=app_payload, headers=headers)
with urllib.request.urlopen(req) as r:
    application = json.loads(r.read())
APP_ID = application['id']
print(f'\n📝 Application ID: {APP_ID}')

# Start a browser automation session
# target_url overrides the grant's stored URL — the bot goes to TEST_GRANT_URL
print(f'\n🤖 Launching Playwright browser → {TEST_GRANT_URL}')
browser_payload = json.dumps({
    'application_id': APP_ID,
    'target_url': TEST_GRANT_URL
}).encode()
req = urllib.request.Request(f'{BASE}/browser/start', data=browser_payload, headers=headers)
with urllib.request.urlopen(req) as r:
    session = json.loads(r.read())
SESSION_ID = session.get('id')
print(f'   Session ID: {SESSION_ID}')
print('⏳ Automation running (polling every 5s, max 2 min)...')

# Poll for completion
result = {}
for tick in range(24):
    time.sleep(5)
    req = urllib.request.Request(f'{BASE}/browser/session/{SESSION_ID}', headers=headers)
    with urllib.request.urlopen(req) as r:
        result = json.loads(r.read())
    st = result.get('status', 'unknown')
    filled = result.get('completed_fields_count', 0)
    print(f'   [{(tick+1)*5:3d}s] Status: {st} | Fields filled: {filled}')
    if st in ('COMPLETED', 'WAITING_FOR_USER', 'FAILED', 'CANCELLED'):
        break

# Report results
print()
print('='*65)
print('📊 BROWSER AUTOMATION RESULT')
print('='*65)
print(f'  Final status       : {result.get("status")}')
print(f'  Fields filled      : {result.get("completed_fields_count", 0)} / {result.get("total_fields_detected", "?")}')
print(f'  Current URL        : {result.get("current_url", "n/a")}')
print(f'  Current task       : {result.get("current_task", "n/a")}')

if result.get('intervention_required'):
    print()
    print('  ⚠️  HUMAN INTERVENTION REQUIRED')
    print(f'     Reason     : {result.get("current_task")}')
    print(f'     Intervention type: {result.get("intervention_type")}')
    shot = result.get('latest_screenshot_path')
    if shot:
        print(f'     Screenshot : {shot}')
        print('     (This file is in your Google Drive screenshots folder)')
    print()
    print('  What to do next:')
    print('    1. Open the screenshot from your Drive to see what the page looks like')
    print('    2. Open the URL manually in your browser and complete the CAPTCHA / OTP')
    print('    3. Come back and POST to /browser/{session_id}/command with action="resume"')

print('='*65)

In [ ]:
# ================================================================
# CELL 6 — Discover a New Grant by Scraping Any URL
#
# What it does:
#   You paste a URL to any grant program page.
#   The Grant Scout agent:
#     1. Fetches the page content
#     2. Sends it to Gemini AI to extract:
#        - Grant name and funder
#        - Funding amount (min and max)
#        - Application deadline
#        - Eligible countries and regions
#        - Required org stage (idea / prototype / MVP / traction)
#        - Whether incorporation is required
#        - Application process description
#        - Required documents list
#        - Application questions
#     3. Classifies the stage as:
#        GREEN  — idea stage accepted (no MVP needed)
#        YELLOW — some validation required
#        ORANGE — MVP required
#        RED    — paying customers / revenue required
#     4. Sets verification status:
#        OFFICIAL_VERIFIED   — confirmed it's a real grant portal
#        UNCONFIRMED         — couldn't fully verify, treat with caution
#     5. Saves the grant to your database
#
# After this cell, you can:
#   - Run Cell 5 to immediately attempt applying to this grant
#   - Check the grant in the web frontend's Discover page
#   - Run the eligibility matcher via POST /api/v1/matching/run
#
# HOW TO USE:
#   Replace GRANT_PAGE_URL with the URL of a real grant program page.
#   It does NOT have to be the application form URL —
#   a program description page works fine.
# ================================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── CONFIGURE THIS ────────────────────────────────────────────────
GRANT_PAGE_URL = 'https://www.tonyelumelufoundation.org/entrepreneurship'
# ^ Paste any real grant program page URL here
# ─────────────────────────────────────────────────────────────────

# Login
login_data = urllib.parse.urlencode({'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

print(f'🔍 Scraping: {GRANT_PAGE_URL}')
print('   Fetching page → sending to Gemini AI → extracting grant details...')
print('   (This takes 5–15 seconds depending on page size and AI response time)')

payload = json.dumps({'url': GRANT_PAGE_URL}).encode()
req = urllib.request.Request(f'{BASE}/grants/scrape', data=payload, headers=headers)
with urllib.request.urlopen(req) as r:
    result = json.loads(r.read())

stage_emoji = {
    'GREEN_IDEA': '🟢', 'GREEN_MVP': '🟢',
    'YELLOW_VALIDATION': '🟡',
    'ORANGE_MVP': '🟠',
    'RED_TRACTION': '🔴'
}.get(result.get('stage_classification', ''), '⚪')

print()
print('='*65)
print('🎯 GRANT DISCOVERED AND ADDED TO YOUR DATABASE')
print('='*65)
print(f'  Grant name     : {result.get("grant_name", "(not extracted)")}')
print(f'  Funder         : {result.get("funder", "(not extracted)")}')
amt_min = result.get("funding_amount_min")
amt_max = result.get("funding_amount_max")
currency = result.get("currency", "USD")
if amt_min and amt_max:
    print(f'  Funding amount : ${amt_min:,.0f} – ${amt_max:,.0f} {currency}')
elif amt_max:
    print(f'  Funding amount : up to ${amt_max:,.0f} {currency}')
else:
    print(f'  Funding amount : (not specified on page)')
print(f'  Deadline       : {result.get("deadline", "(not found)")}')
print(f'  Stage class    : {stage_emoji} {result.get("stage_classification", "unknown")}')
print(f'  App status     : {result.get("application_status", "unknown")}')
print(f'  Verification   : {result.get("verification_status", "unknown")}')
print(f'  Countries      : {result.get("eligible_countries", [])}')
print(f'  MVP required   : {result.get("mvp_required", "?")}')
print(f'  Incorporation  : {result.get("incorporation_required", "?")}')
print(f'  Grant ID       : {result.get("id")}')
print()
if result.get('required_questions'):
    print(f'  Application questions ({len(result["required_questions"])} found):')
    for i, q in enumerate(result['required_questions'][:3], 1):
        print(f'    {i}. {q[:100]}...' if len(q) > 100 else f'    {i}. {q}')
    if len(result['required_questions']) > 3:
        print(f'    ... and {len(result["required_questions"]) - 3} more')
print()
print('  ✅ Grant is now in your database.')
print('     Run Cell 5 with this grant URL to attempt the application.')
print('='*65)

In [ ]:
# ================================================================
# CELL 7 — Deploy Grant Agent to Google Cloud Run (Production)
#
# WHY USE THIS:
#   Colab sessions end after 12–24 hours. If you want Grant Agent
#   running 24/7 with no session limits, you deploy it to
#   Google Cloud Run. Cloud Run keeps it always-on and you only
#   pay for actual compute time (usually under $5/month for light use).
#
# WHAT IT DEPLOYS:
#   Three separate Cloud Run services:
#
#   1. grant-agent-api   (the FastAPI backend)
#      - Public HTTPS URL (you can call it from anywhere)
#      - 2 vCPU, 2 GB RAM
#      - Min 1 instance (always warm, no cold start)
#      - Secrets pulled from Google Secret Manager (DB URL, JWT, Gemini key)
#
#   2. grant-agent-worker  (the Playwright browser automation worker)
#      - Private (only the API can call it)
#      - 4 vCPU, 4 GB RAM (Chromium needs this)
#      - 15-minute timeout per job (long grant application forms)
#      - Scales to 0 when idle (saves money)
#
#   3. grant-agent-web   (the Next.js frontend)
#      - Public HTTPS URL
#      - 1 vCPU, 1 GB RAM
#      - Automatically wired to the API URL
#
#   PLUS:
#   - Cloud Scheduler: runs the grant scout every day at 2 AM UTC
#     (automatically discovers new open grants for your org)
#   - Cloud Artifact Registry: stores the Docker container images
#
# PRE-REQUISITES (do these once before running this cell):
#   1. Create a Google Cloud project at https://console.cloud.google.com
#   2. Enable billing on it (Cloud Run has a free tier but billing must be on)
#   3. Set GCP_PROJECT_ID in Cell 2 to your real project ID
#   4. Run the other Cloud setup scripts first (in deploy/ folder):
#        gcp-cloud-sql.sh     — creates PostgreSQL database
#        gcp-secret-manager.sh — stores your secrets
#        gcp-cloud-tasks.sh   — creates the job queue
#      OR skip those and use SQLite on Cloud Run (simpler, fine for testing)
#
# HOW TO USE THIS CELL:
#   1. Make sure GCP_PROJECT_ID is set correctly in Cell 2
#   2. Run this cell
#   3. A browser window will pop up for Google authentication — sign in
#   4. Wait ~10-15 minutes for all 3 Docker images to build and deploy
#   5. At the end you get two permanent URLs:
#        API URL : https://grant-agent-api-xxxx-uc.a.run.app
#        Web URL : https://grant-agent-web-xxxx-uc.a.run.app
# ================================================================

import subprocess, sys, os, time

REPO_DIR   = '/content/grant-agent'
PROJECT_ID = GCP_PROJECT_ID    # from Cell 2
REGION     = GCP_REGION        # from Cell 2 (default: us-central1)
AR_REPO    = 'grant-agent-repo'

def run(cmd, **kwargs):
    """Run a shell command, print output live, raise on failure."""
    print(f'\n$ {cmd}')
    proc = subprocess.run(cmd, shell=True, text=True,
                          capture_output=False, **kwargs)
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc

# ── Step 0: Install gcloud CLI ────────────────────────────────────
print('='*65)
print('STEP 0: Installing / verifying gcloud CLI')
print('='*65)

result = subprocess.run('which gcloud', shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print('Installing gcloud CLI (~300 MB, takes ~2 minutes)...')
    run('curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts --install-dir=/usr/local')
    os.environ['PATH'] += ':/usr/local/google-cloud-sdk/bin'
    run('gcloud --version')
else:
    print('✅ gcloud already installed:', result.stdout.strip())
    run('gcloud --version')

# ── Step 1: Authenticate with Google ─────────────────────────────
print()
print('='*65)
print('STEP 1: Google Cloud Authentication')
print('='*65)
print('A browser window will open. Sign in with the Google account')
print('that owns your Cloud project.')
print()
run('gcloud auth login --no-launch-browser')
# The --no-launch-browser flag prints a URL to paste in your browser
# and a code to paste back. Follow the prompts.

run(f'gcloud config set project {PROJECT_ID}')
run('gcloud config set core/disable_usage_reporting true')
print(f'✅ Authenticated and project set to: {PROJECT_ID}')

# ── Step 2: Enable required GCP APIs ─────────────────────────────
print()
print('='*65)
print('STEP 2: Enabling required Google Cloud APIs')
print('(takes ~1 minute the first time)')
print('='*65)

apis = [
    'run.googleapis.com',           # Cloud Run
    'cloudbuild.googleapis.com',    # Build Docker images
    'artifactregistry.googleapis.com',  # Store Docker images
    'secretmanager.googleapis.com', # Store secrets securely
    'cloudscheduler.googleapis.com' # Daily grant discovery cron
]
run(f'gcloud services enable {" ".join(apis)} --project={PROJECT_ID}')
print('✅ All APIs enabled')

# ── Step 3: Create Artifact Registry repository ───────────────────
print()
print('='*65)
print('STEP 3: Creating Docker image registry')
print('='*65)

run(f'''gcloud artifacts repositories create {AR_REPO} \
    --repository-format=docker \
    --location={REGION} \
    --description="Docker images for Grant Agent" \
    --project={PROJECT_ID} || echo "Registry already exists, continuing."''')

run(f'gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet')
print('✅ Artifact Registry ready')

# ── Step 4: Build and push Docker images ─────────────────────────
print()
print('='*65)
print('STEP 4: Building Docker images with Cloud Build')
print('(This takes 5–10 minutes — gcloud sends the repo to Google servers to build)')
print('='*65)

IMG_BASE = f'{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}'

print('\n📦 Building API image...')
run(f'gcloud builds submit --tag {IMG_BASE}/grant-agent-api:latest -f deploy/Dockerfile.api .', cwd=REPO_DIR)

print('\n📦 Building Worker image (includes Chromium — largest image)...')
run(f'gcloud builds submit --tag {IMG_BASE}/grant-agent-worker:latest -f deploy/Dockerfile.worker .', cwd=REPO_DIR)

print('\n📦 Building Web (Next.js) image...')
run(f'gcloud builds submit --tag {IMG_BASE}/grant-agent-web:latest -f deploy/Dockerfile.web .', cwd=REPO_DIR)

print('\n✅ All 3 Docker images built and pushed to Artifact Registry')

# ── Step 5: Deploy API to Cloud Run ──────────────────────────────
print()
print('='*65)
print('STEP 5: Deploying FastAPI backend to Cloud Run')
print('='*65)

run(f'''gcloud run deploy grant-agent-api \
    --image={IMG_BASE}/grant-agent-api:latest \
    --region={REGION} \
    --platform=managed \
    --allow-unauthenticated \
    --cpu=2 --memory=2Gi \
    --min-instances=1 --max-instances=10 \
    --set-env-vars="ENVIRONMENT=production" \
    --set-secrets="DATABASE_URL=grant-agent-db-url:latest,SECRET_KEY=grant-agent-jwt-secret:latest,GEMINI_API_KEY=grant-agent-gemini-key:latest" \
    --project={PROJECT_ID}''')

# Get the API URL
api_url_result = subprocess.run(
    f'gcloud run services describe grant-agent-api --region={REGION} --format="value(status.url)" --project={PROJECT_ID}',
    shell=True, capture_output=True, text=True
)
API_URL = api_url_result.stdout.strip()
print(f'\n✅ API deployed at: {API_URL}')

# ── Step 6: Deploy Worker to Cloud Run ───────────────────────────
print()
print('='*65)
print('STEP 6: Deploying Playwright browser worker to Cloud Run')
print('(Worker is private — only the API can call it)')
print('='*65)

run(f'''gcloud run deploy grant-agent-worker \
    --image={IMG_BASE}/grant-agent-worker:latest \
    --region={REGION} \
    --platform=managed \
    --no-allow-unauthenticated \
    --cpu=4 --memory=4Gi \
    --timeout=900 \
    --min-instances=0 --max-instances=5 \
    --set-secrets="DATABASE_URL=grant-agent-db-url:latest,SECRET_KEY=grant-agent-jwt-secret:latest" \
    --project={PROJECT_ID}''')
print('✅ Browser worker deployed (scales to 0 when idle)')

# ── Step 7: Deploy Web Frontend to Cloud Run ──────────────────────
print()
print('='*65)
print('STEP 7: Deploying Next.js frontend to Cloud Run')
print('='*65)

run(f'''gcloud run deploy grant-agent-web \
    --image={IMG_BASE}/grant-agent-web:latest \
    --region={REGION} \
    --platform=managed \
    --allow-unauthenticated \
    --cpu=1 --memory=1Gi \
    --min-instances=1 --max-instances=10 \
    --set-env-vars="NEXT_PUBLIC_API_URL={API_URL}/api/v1" \
    --project={PROJECT_ID}''')

web_url_result = subprocess.run(
    f'gcloud run services describe grant-agent-web --region={REGION} --format="value(status.url)" --project={PROJECT_ID}',
    shell=True, capture_output=True, text=True
)
WEB_URL = web_url_result.stdout.strip()
print(f'✅ Frontend deployed at: {WEB_URL}')

# ── Step 8: Cloud Scheduler — daily grant discovery ───────────────
print()
print('='*65)
print('STEP 8: Setting up Cloud Scheduler (daily grant discovery)')
print('Every day at 2 AM UTC, the AI scout automatically searches')
print('for new open grants matching your organisation profiles.')
print('='*65)

run(f'''gcloud scheduler jobs create http grant-agent-daily-scout \
    --schedule="0 2 * * *" \
    --uri="{API_URL}/api/v1/automation/scout-run" \
    --http-method=POST \
    --time-zone="Etc/UTC" \
    --location={REGION} \
    --project={PROJECT_ID} || echo "Scheduler job already exists."''')
print('✅ Cloud Scheduler set (runs every day at 02:00 UTC)')

# ── Final summary ─────────────────────────────────────────────────
print()
print('='*65)
print('🎉  DEPLOYMENT COMPLETE — Grant Agent is running 24/7!')
print('='*65)
print(f'  🌐  Web Platform  : {WEB_URL}')
print(f'  📡  API Endpoint  : {API_URL}')
print(f'  📖  API Docs      : {API_URL}/docs')
print(f'  ❤️   Health check  : {API_URL}/health')
print()
print(f'  🔑  Login credentials:')
print(f'      Email    : demo@grantagent.ai')
print(f'      Password : password123')
print()
print(f'  ⏰  Daily grant scout runs at 02:00 UTC every day')
print(f'  📁  GCP project   : {PROJECT_ID}')
print(f'  🗺️   Region        : {REGION}')
print()
print('  Next steps:')
print('    1. Open the Web Platform URL and log in')
print('    2. Go to Organisations and update your profile with real data')
print('    3. Go to Discover and paste a real grant URL to scrape it')
print('    4. The AI will match grants to your org automatically')
print('    5. Click Apply to launch the browser bot on any matched grant')
print('='*65)